# Classes Abstratas e Interfaces em Python


## Relembrando: herança entre classes concretas

Uma classe é **concreta** quando ela pode ser instanciada, ou seja, quando podemos criar objetos a partir dela normalmente. Quando a superclasse **e** a subclasse são concretas, a herança serve basicamente para **reaproveitar** atributos e métodos, e a subclasse pode:

- usar os métodos da superclasse sem mudar nada;
- **sobrescrever** um método, mudando o comportamento;
- adicionar novos métodos e atributos que só ela possui.

**Exemplo:** uma classe `Senha`, que sabe validar uma senha e mostrar as regras de criação. E uma subclasse `SenhaCriptografada`, que faz a mesma coisa, mas precisa criptografar a senha antes de guardar ou comparar com o banco de dados.

In [ ]:
class Senha:
    def __init__(self, valor):
        self.valor = valor  # senha armazenada (aqui, em texto puro)

    def validar(self, senha_digitada):
        # compara a senha digitada com o valor armazenado
        pass

    def exibir_regras(self):
        # imprime na tela as regras de criação de senha
        # (tamanho mínimo, uso de números, letras maiúsculas, etc.)
        pass


class SenhaCriptografada(Senha):
    def __init__(self, valor):
        super().__init__(valor)

    def criptografar(self, senha_texto_puro):
        # transforma uma senha em texto puro em uma versão criptografada,
        # para ser enviada e guardada no banco de dados
        pass

    def validar(self, senha_digitada):
        # sobrescreve o método da superclasse: antes de comparar,
        # é preciso criptografar a senha digitada, já que no banco
        # só existe a versão criptografada
        pass


Repare que `SenhaCriptografada` **reaproveita** `exibir_regras()` sem nenhuma alteração, mas **sobrescreve** `validar()` para incluir a criptografia antes da comparação.

E o mais importante: tanto `Senha` quanto `SenhaCriptografada` podem virar objetos de verdade no nosso sistema (`s = Senha("1234")` funciona normalmente). É isso que caracteriza uma herança entre classes **concretas**.

## Quando a superclasse nunca é instanciada

Nem toda herança segue esse padrão. Às vezes criamos uma superclasse só para **organizar** a hierarquia e reaproveitar código, mas **nunca** temos a intenção de criar um objeto diretamente dela.

**Exemplo:** em um sistema de RH, todo `Funcionario` tem um **cargo**, ele é Vendedor, é Gerente, é Estagiário... nunca existe um Funcionário genérico, sem cargo nenhum. Ou seja, ninguém deveria escrever `f = Funcionario("Ana", 3000)`.

Mesmo assim, começamos com métodos **concretos** (já implementados) na superclasse, porque eles são iguais para todo mundo, independente do cargo.

In [ ]:
class Funcionario:
    def __init__(self, nome, salario_base):
        self.nome = nome
        self.salario_base = salario_base

    def exibir_dados(self):
        # imprime o nome e o cargo do funcionário
        pass

    def bater_ponto(self):
        # registra o horário de entrada/saída do funcionário
        pass


class Vendedor(Funcionario):
    def __init__(self, nome, salario_base, taxa_comissao):
        super().__init__(nome, salario_base)
        self.vendas = []
        self.taxa_comissao = taxa_comissao

    def registra_venda(self):
        # Associa uma nova venda a esse funcionário para cálculo de comissão
        pass


class Gerente(Funcionario):
    def __init__(self, nome, salario_base, beneficios):
        super().__init__(nome, salario_base)
        self.beneficios = []

    def adiciona_beneficio(self):
        # Adiciona um novo benefício financeiro
        pass


Só que aqui existe um problema: **nada no código impede** alguém de escrever `Funcionario("Ana", 2000)`. O Python deixa, porque `Funcionario` é uma classe concreta como qualquer outra, a regra de "todo funcionário tem cargo" existe só na nossa cabeça, não no código.

É exatamente esse o conceito que uma **classe abstrata** é capaz de lidar.

## Classe Abstrata

**Definição formal:** uma classe abstrata é uma classe que **não pode ser instanciada diretamente**. Ela existe apenas para servir de modelo (molde) para suas subclasses. Uma classe abstrata pode ter:

- **métodos concretos**, com implementação normal, que são herdados como sempre;
- **métodos abstratos**, que têm apenas a assinatura (nome, parâmetros), sem implementação. Toda subclasse concreta é **obrigada** a implementar esses métodos.

De forma mais simples: um método abstrato serve para dizer que toda subclasse precisa ter esse método, mas não faz sentido definir um comportamento padrão.

Em Python, usamos o módulo `abc` (*Abstract Base Classes*): a classe herda de `ABC` (*Abstract Base Class*) e os métodos abstratos recebem o decorador `@abstractmethod`.

**Exemplo:** todo `Funcionario` precisa saber calcular seu próprio salário (`calcula_salario`), mas a forma de calcular é bem diferente entre um `Gerente` (horas trabalhadas + benefícios) e um `Vendedor` (vendas + comissão).

In [ ]:
from abc import ABC, abstractmethod

class Funcionario(ABC):
    def __init__(self, nome):
        self.nome = nome

    def exibir_dados(self):
        # método concreto: todo funcionário exibe os dados do mesmo jeito
        pass

    @abstractmethod
    def calcula_salario(self):
        # método abstrato: toda subclasse PRECISA implementar,
        # mas cada uma calcula o salário de um jeito diferente
        pass # nesse caso o "pass" é necessário!!!


class Vendedor(Funcionario):
    def __init__(self, nome, salario_fixo, taxa_comissao):
        super().__init__(nome)
        self.salario_fixo = salario_fixo
        self.vendas = []
        self.taxa_comissao = taxa_comissao

    def registra_venda(self):
        # Associa uma nova venda a esse funcionário para cálculo de comissão
        pass

    def calcula_salario(self):
        # salário = salário fixo + comissão sobre o total vendido
        pass


class Gerente(Funcionario):
    def __init__(self, nome, horas_trabalhadas, valor_hora):
        super().__init__(nome)
        self.horas_trabalhadas = horas_trabalhadas
        self.valor_hora = valor_hora
        self.beneficios = []

    def adiciona_beneficio(self):
        # Adiciona um novo benefício financeiro
        pass

    def calcula_salario(self):
        # salário = horas trabalhadas * valor da hora + valor dos benefícios
        pass


Agora, diferente do exemplo anterior, o Python **impede** de verdade a criação de um `Funcionario` "genérico". Se tentarmos instanciar a classe abstrata diretamente, recebemos um erro:

In [ ]:
f = Funcionario("Ana")

TypeError: Can't instantiate abstract class Funcionario without an implementation for abstract method 'calcula_salario'

## Interface

Uma **interface** é o caso extremo da classe abstrata: é uma classe **totalmente abstrata**, onde **todos** os métodos são abstratos, não existe nenhum método concreto, nenhuma implementação padrão.

Uma interface não diz *"como fazer"*, ela só diz *"o que precisa existir"*. É como um **contrato**: quem implementa a interface promete ter aqueles métodos, mas cada implementação faz do seu próprio jeito, do zero.

> Pense assim: a interface é uma ficha de requisitos. *"Para ser considerado um Funcionario neste sistema, você precisa saber exibir seus dados, bater ponto e calcular seu salário."* A interface não dá nenhuma dica de como fazer isso, só exige que exista.

Python não tem uma palavra-chave `interface` como outras linguagens (Java, por exemplo). Podemos fazer isso de diversas maneiras, a mais simples delas é criando uma classe `ABC` em que **todos** os métodos são `@abstractmethod`.

Vamos redefinir `Funcionario` como uma interface:

In [ ]:
from abc import ABC, abstractmethod

class Funcionario(ABC):
    @abstractmethod
    def exibir_dados(self):
        # define que toda subclasse precisa exibir seus dados,
        # mas não diz como fazer isso
        pass

    @abstractmethod
    def bater_ponto(self):
        # define que toda subclasse precisa registrar o ponto,
        # mas não diz como fazer isso
        pass

    @abstractmethod
    def calcula_salario(self):
        # define que toda subclasse precisa calcular seu próprio salário
        pass


class Gerente(Funcionario):
    def __init__(self, nome, horas_trabalhadas, valor_hora, beneficios):
        self.nome = nome
        self.horas_trabalhadas = horas_trabalhadas
        self.valor_hora = valor_hora
        self.beneficios = beneficios

    def exibir_dados(self):
        pass # aqui teria implementação concreta!!

    def bater_ponto(self):
        pass # aqui teria implementação concreta!!

    def calcula_salario(self):
        pass # aqui teria implementação concreta!!

    # método não previsto na Interface (tudo bem)
    def adiciona_beneficio(self):
        # Adiciona um novo benefício financeiro
        pass


class Vendedor(Funcionario):
    def __init__(self, nome, salario_fixo, vendas, taxa_comissao):
        self.nome = nome
        self.salario_fixo = salario_fixo
        self.vendas = vendas
        self.taxa_comissao = taxa_comissao

    def exibir_dados(self):
        pass # aqui teria implementação concreta!!

    def bater_ponto(self):
        pass # aqui teria implementação concreta!!

    def calcula_salario(self):
        pass # aqui teria implementação concreta!!

    # método não previsto na Interface (tudo bem)
    def registra_venda(self):
        # Associa uma nova venda a esse funcionário para cálculo de comissão
        pass


Note que agora **nenhum** método veio "pronto" de `Funcionario`. Cada subclasse (`Gerente`, `Vendedor`) precisa implementar `exibir_dados`, `bater_ponto` e `calcula_salario` do zero, do seu próprio jeito. `Funcionario` só garante que esses três métodos vão existir em qualquer classe que a implemente.

# Exercícios de laboratório

## Q1.
Implemente as classes envolvidas no problema **Entregas de uma transportadora**.

- **`Entrega`** — atributos: `endereco_destino`, `status` e `peso`. Métodos:
    - `atualizar_status()`: Recebe uma string "status" ("em separação", "a caminho" ou "entregue") e atualiza o atributo correspondente da classe.
    - `imprimir_dados()`: Imprime todos os atributos do pedido.
    - `calcular_frete()`: abstrata

- **`EntregaTerrestre(Entrega)`** — atributo: `distancia_km`. Métodos:
    - `imprimir_dados()`: Chama o método correspondente da superclasse, imprime seu atributo interno `distancia_km` e o resultado do método `calcular_frete()`.
    - `calcular_frete()`: Cobra de acordo com o atributo `distancia_km`: Até 100 km: taxa fixa de **20 reais**; De 101 km até 500 km: **40 reais** fixos **+ 1,50 por kg**; Acima de 500 km: **70 reais** fixos **+ 2,20 por kg**.
- **`EntregaAerea(Entrega)`** — atributo: `taxa_despacho`. Métodos:
    - `imprimir_dados()`: Chama o método correspondente da superclasse, imprime seu atributo interno `taxa_despacho` e o resultado do método `calcular_frete()`.
    - `calcular_frete()`: Cobra a `taxa_despacho` fixa de **+ 12 reais por kg**.

- `main.py`: Implemente no programa principal uma lista de entregas e um menu com as opções:
    - Criar nova entrega: o usuário escolhe entre entrega terrestre ou aérea e fornece os atributos correspondentes. A entrega é adicionada na lista de entregas do programa principal.
    - Listar entregas: chama o método `imprimir_dados()` de todas as entregas na lista do programa principal.



In [ ]:
# CLASSES
class Entrega:
    def __init__(self, endereco_destino, status, peso):
        self.endereco_destino = endereco_destino
        self.status = status
        self.peso = peso
    def atualizar_status(self):
        self.status = str(input("Informe a situação do produto:\nEm separação\nA caminho\nEntregue\n"))
    def imprimir_dados(self):
        print(self.endereco_destino, self.status, self.peso)
    def calcular_frete(self):
        pass
class EntregaTerrestre(Entrega):
    def __init__(self, endereco_destino, status, peso, distancia_km):
        super().__init__(endereco_destino, status, peso)
        self.distancia_km = distancia_km
    def calcular_frete(self):
        self.distancia_km = float(input("Informe a distância em KM: "))
        self.peso = float(input("Informe o peso do pacote em KG: "))
        if self.distancia_km <= 100:
            print("O frete é de 20 Reais!")
        elif self.distancia_km <= 500:
            add = self.peso * 1.50 + 40
            print(f"O frete é de {add} Reais!")
        elif self.distancia_km > 500:
            add = self.peso * 2.20 + 70
            print(f"O frete é de {add} Reais!")
    def imprimir_dados(self):
        print(self.endereco_destino, self.status, self.peso, self.distancia_km)
class EntregaAerea(Entrega):
    def __init__(self, endereco_destino, status, peso, taxa_despacho):
        super().__init__(endereco_destino, status, peso)
        self.taxa_despacho = taxa_despacho
    def calcular_frete(self):
        self.taxa_despacho = 12 * self.peso
    def imprimir_dados(self):
        print(self.endereco_destino, self.status, self.peso, self.taxa_despacho)
# MAIN
from Q1 import Entrega, EntregaAerea, EntregaTerrestre
import os
os.system("cls")

entregas = []

while True:
    op = int(input(
        "O que deseja fazer?\n"
        "1 - Criar nova Entrega\n"
        "2 - Listar Entregas\n"
        "3 - Alterar Status de Entrega\n"
        "4 - Sair\n"))
    if op == 1:
        op1 = int(input(
            "Qual tipo de entrega deseja?\n"
            "1 - Entrega Terrestre\n"
            "2 - Entrega Aérea\n" ))
        if op1 == 1:
            endereco_destino = input("Informe o endereço de destino: ")
            peso = float(input("Informe o peso do pacote em kg: "))
            distancia_km = float(input("Informe a distância em km: "))
            status = "Em separação"
            obj = EntregaTerrestre(endereco_destino, status, peso, distancia_km)
            entregas.append(obj)
            cod = len(entregas)
            print(f"O Código dessa entrega é {cod}")
        elif op1 == 2:
            endereco_destino = input("Informe o endereço de destino: ")
            peso = float(input("Informe o peso do pacote em kg: "))
            taxa_despacho = 12 * peso
            status = "Em separação"
            obj = EntregaAerea(endereco_destino, status, peso, taxa_despacho)
            entregas.append(obj)
            cod = len(entregas)
            print(f"O Código dessa entrega é {cod}")
        else:
            print("Opção inválida!")
    elif op == 2:
        if len(entregas) == 0:
            print("Não existem entregas cadastradas.")
        else:
            for i, entrega in enumerate(entregas):
                print(f"\n--- Entrega {i + 1} ---")
                entrega.imprimir_dados()
    elif op == 3:
        if len(entregas) == 0:
            print("Não existem entregas cadastradas.")
        else:
            cod = int(input("Informe o código da entrega: "))
            if cod >= 1 and cod <= len(entregas):
                entregas[cod - 1].atualizar_status()
                print("Status atualizado com sucesso!")
            else:
                print("Código de entrega inválido!")
    elif op == 4:
        print("Programa encerrado!")
        break
    else:
        print("Opção inválida!")

## Q2.

Vamos fazer uma versão simples do problema **Meios de pagamento de um checkout**.

- **`MeioDePagamento`** — atributo `status`, e métodos **abstratos**: `processar_pagamento()`, `cancelar_pagamento()`, `gerar_comprovante()`.
- **`CartaoCredito(MeioDePagamento)`** — atributo `numero_cartao`. Métodos:
    - processar_pagamento(): Altera o `status` para "aprovado" e imprime uma mensagem informando que o pagamento foi processado via cartão de crédito
    - cancelar_pagamento(): Altera o `status` para "cancelado" e imprime uma mensagem informando que o pagamento foi cancelado.
    - gerar_comprovante(): Caso o `status` seja "aprovado", imprime a frase "comprovante gerado para o carão número: " e **imprime os últimos 4 dígitos do cartão de crédito**. Caso o `status` seja "cancelado" apenas informa o usuário que o pagamento foi cancelado.
- **`Pix(MeioDePagamento)`** — atributo: `chave_pix`. Métodos:
    - processar_pagamento(): Altera o `status` para "aprovado", imprime a mensagem "Aguardando pagamento para a seguinte chave Pix: " e imprime o atributo `chave_pix`.
    - cancelar_pagamento(): Imprime que a operação Pix não pode ser cancelada.
    - gerar_comprovante(): imprime a frase "comprovante gerado".
- **`Boleto(MeioDePagamento)`** — atributo: `codigo_de_barras`. Métodos:
    - processar_pagamento(): Altera o `status` para "aprovado", imprime a mensagem "Aguardando pagamento para o boleto de código:" e imprime o atributo `codigo_de_barras`.
    - cancelar_pagamento(): Altera o `status` para "cancelado e imprime "Boleto cancelado".
    - gerar_comprovante(): Caso o `status` seja "aprovado", imprime a frase "comprovante gerado". Caso o `status` seja "cancelado" apenas informa o usuário que o pagamento foi cancelado.

- `main.py`: No programa principal, pergunte ao usuário qual o método de pagamento, crie o objeto correspondente, e apresente um menu onde cada opção corresponde à chamada de um dos métodos das classes.




In [ ]:
# CLASSES
from abc import ABC, abstractmethod

class MeioDePagamento(ABC):
    def __init__(self, status):
        self.status = status

    @abstractmethod
    def processar_pagamento(self):
        pass

    @abstractmethod
    def cancelar_pagamento(self):
        pass

    @abstractmethod
    def gerar_comprovante(self):
        pass

class CartaoCredito(MeioDePagamento):
    def __init__(self, status, numero_cartao):
        super().__init__(status)
        self.numero_cartao = numero_cartao

    def processar_pagamento(self):
        self.status = "aprovado"
        print("Pagamento processado via cartão de crédito")

    def cancelar_pagamento(self):
        self.status = "cancelado"
        print("Pagamento cancelado")

    def gerar_comprovante(self):
        if self.status == "aprovado":
            print("Comprovante gerado para o cartão número:")
            print(self.numero_cartao[-4:])
        elif self.status == "cancelado":
            print("O pagamento foi cancelado")

class Pix(MeioDePagamento):
    def __init__(self, status, chave_pix):
        super().__init__(status)
        self.chave_pix = chave_pix

    def processar_pagamento(self):
        self.status = "aprovado"
        print("Aguardando pagamento para a seguinte chave Pix:")
        print(self.chave_pix)

    def cancelar_pagamento(self):
        print("A operação Pix não pode ser cancelada")

    def gerar_comprovante(self):
        print("Comprovante gerado")

class Boleto(MeioDePagamento):
    def __init__(self, status, codigo_de_barras):
        super().__init__(status)
        self.codigo_de_barras = codigo_de_barras

    def processar_pagamento(self):
        self.status = "aprovado"
        print("Aguardando pagamento para o boleto de código:")
        print(self.codigo_de_barras)

    def cancelar_pagamento(self):
        self.status = "cancelado"
        print("Boleto cancelado")

    def gerar_comprovante(self):
        if self.status == "aprovado":
            print("Comprovante gerado")
        elif self.status == "cancelado":
            print("O pagamento foi cancelado")
  # MAIN
  from Q1 import MeioDePagamento, CartaoCredito, Pix, Boleto
import os

os.system("cls")

op = int(input("Qual método de pagamento deseja usar?\n1 - Cartão de Crédito\n2 - Pix\n3 - Boleto\n"))

status = "pendente"

if op == 1:
    numero_cartao = input("Informe o número do cartão: ")
    obj = CartaoCredito(status, numero_cartao)

elif op == 2:
    chave_pix = input("Informe a chave Pix: ")
    obj = Pix(status, chave_pix)

elif op == 3:
    codigo_de_barras = input("Informe o código de barras: ")
    obj = Boleto(status, codigo_de_barras)

else:
    print("Opção inválida")
    exit()

while True:
    op1 = int(input("\nO que deseja fazer?\n1 - Processar pagamento\n2 - Cancelar pagamento\n3 - Gerar comprovante\n4 - Sair\n"))

    if op1 == 1:
        obj.processar_pagamento()

    elif op1 == 2:
        obj.cancelar_pagamento()

    elif op1 == 3:
        obj.gerar_comprovante()

    elif op1 == 4:
        print("Programa encerrado!")
        break

    else:
        print("Opção inválida!")

---
# Exercício de sala: minimundos

Para cada minimundo abaixo, leia a descrição do sistema e decida, para cada método listado:

- Ele deveria ser **concreto** (com implementação padrão)?
- Ele deveria ser **abstrato** (só a assinatura, cada subclasse implementa do seu jeito)?
- E a superclasse: é **concreta**, **abstrata** ou uma **interface**?



## Pedidos de uma loja online

Em um sistema de e-commerce, todo pedido feito por um cliente vira um objeto `Pedido`, mesmo que não tenha nenhum desconto aplicado. Quando o cliente usa um cupom, o pedido passa a ser tratado como `PedidoComDesconto`, que reaproveita as mesmas informações de um pedido comum, mas aplica o abatimento do cupom antes de fechar o valor final. A loja continua vendendo pedidos "normais", sem cupom, o tempo todo.

**Classes envolvidas:**

- **`Pedido`** — atributos: `itens`, `cliente`. Métodos: `calcular_subtotal()` (soma o valor dos itens), `gerar_nota_fiscal()` (monta o documento fiscal do pedido), `fechar_pedido()` (finaliza a compra).
- **`PedidoComDesconto(Pedido)`** — atributo: `codigo_cupom`. Método: `aplicar_cupom()` (desconta o valor do cupom sobre o subtotal antes de fechar), **.... (mais algum método?)**


## Entregas de uma transportadora

Uma transportadora registra todo pedido despachado como uma `Entrega`. Toda entrega tem um endereço de destino e um peso, e precisa atualizar seu status durante o trajeto (ex: "em trânsito", "entregue"), e o valor do frete é calculado de formas diferentes dependendo do meio de transporte: entregas terrestres cobram por quilômetro rodado, enquanto entregas aéreas cobram por peso mais uma taxa fixa de despacho no aeroporto.

**Classes envolvidas:**

- **`Entrega`** — atributos: `endereco_destino`, `status` e `peso`. Métodos: `atualizar_status()` (registra a fase atual da entrega), `calcular_frete()` (calcula o valor cobrado pelo transporte).
- **`EntregaTerrestre(Entrega)`** — atributo: `distancia_km`. Métodos: ?
- **`EntregaAerea(Entrega)`** — atributo: `taxa_despacho`. Métodos: ?


## Meios de pagamento de um checkout

Um sistema de checkout precisa aceitar vários meios de pagamento: cartão de crédito, Pix e boleto. Cada meio se comunica com um sistema distinto, o cartão passa por uma operadora de cartões, o Pix gera um QR code através do Banco Central, e o boleto é registrado em um serviço de cobrança bancária.

**Classes envolvidas:**

- **`MeioDePagamento`** — métodos: `processar_pagamento()`, `cancelar_pagamento()`, `gerar_comprovante()`.
- **`CartaoCredito(MeioDePagamento)`** — atributo: `numero_cartao`. Métodos: ?
- **`Pix(MeioDePagamento)`** — atributo: `chave_pix`. Métodos: ?
- **`Boleto(MeioDePagamento)`** — atributo: `linha_digitavel`. Métodos: ?

